[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srini2k03/jobpost-extractor-qlora/blob/main/notebooks/01_paraphrase_7b.ipynb)

# 01 · Paraphrase pass (7B) + real test set drafts

Rewrites a subset of synthetic training postings with **Qwen2.5-7B-Instruct (4-bit)** for realism, keeps only rewrites that pass the rule + NLI checker, then fetches ~200 real public postings and drafts their labels with the same 7B model (few-shot) for human verification with `jobx verify`.

**How to run**
1. Runtime → Change runtime type → **T4 GPU**.
2. Secrets (key icon in the left sidebar), each with *Notebook access* switched on: `HF_TOKEN`, `HF_USERNAME`.
   `HF_USERNAME` is your Hugging Face account name; `HF_TOKEN` needs *write* access.
3. Runtime → **Run all**. Expected runtime: **~1.5 h (paraphrase ~60 min for 600 postings, drafts ~20 min)**. Cells after the setup are safe to re-run.

**Outputs:** `data/paraphrased/paraphrased_train.jsonl` + `filter_stats.json`, `data/real/test_real_draft.jsonl` (+ `raw_meta.json`), all pushed to the private dataset repo `<HF_USERNAME>/jobx-artifacts`.

All code lives in `src/jobx` of the GitHub repo; this notebook only orchestrates it.

In [ ]:
# Setup: clone the repo (code + data), install pinned dependencies, install jobx itself.
import os, sys
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/srini2k03/jobpost-extractor-qlora.git"
if IN_COLAB:
    if not os.path.exists("/content/jobpost-extractor-qlora"):
        !git clone --depth 1 {REPO_URL} /content/jobpost-extractor-qlora
    %cd /content/jobpost-extractor-qlora
    !pip install -q transformers==4.56.2 peft==0.17.1 trl==0.23.1 bitsandbytes==0.48.1 accelerate==1.10.1 datasets==4.1.1 huggingface_hub==0.35.3 wandb==0.22.2 lm-format-enforcer==0.11.3
    !pip install -q -e .
else:
    # running locally from a checkout: notebooks/ -> repo root
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU'
print(torch.__version__, torch.cuda.get_device_name(0))


In [ ]:
# Secrets -> environment variables (read by jobx.hub and wandb).
import os
for key in ("HF_TOKEN", "HF_USERNAME", "WANDB_API_KEY"):
    if not os.environ.get(key):
        try:
            from google.colab import userdata
            os.environ[key] = userdata.get(key)
        except Exception:
            print(f"secret {key} not available")
from jobx.hub import hf_username, repo_id
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("Hub account:", hf_username(), "| artifacts repo:", repo_id("artifacts"))

## Settings

In [ ]:
N_PARAPHRASE = 600      # training postings to rewrite
BATCH = 8
TEACHER = 'Qwen/Qwen2.5-7B-Instruct'
NLI_MODEL = 'cross-encoder/nli-deberta-v3-small'
DO_REAL_DRAFTS = True

## 1. Paraphrase a training subset

In [ ]:
import json, random
from pathlib import Path
from jobx.data.build import read_jsonl, write_jsonl
from jobx.data.paraphrase import STYLES, paraphrase_messages, pick_subset, filter_paraphrases
from jobx.baselines.hf_runner import load_model, generate

train = read_jsonl('data/processed/train.jsonl')
subset = pick_subset(train, N_PARAPHRASE)
rng = random.Random(0)
styles = [rng.choice(STYLES) for _ in subset]
model, tok = load_model(TEACHER, quant='4bit')
outs = []
from tqdm.auto import tqdm
for i in tqdm(range(0, len(subset), BATCH)):
    chunk = subset[i:i + BATCH]
    msgs = [paraphrase_messages(e['text'], s) for e, s in zip(chunk, styles[i:i + BATCH])]
    outs += [g.text for g in generate(model, tok, msgs, max_new_tokens=700, sample=True)]
len(outs)

## 2. Validate: every gold value recoverable, nothing new (rules), entailment (NLI)

In [ ]:
from jobx.data.validate import hf_nli_scorer
nli = hf_nli_scorer(NLI_MODEL, device=0)
kept, stats = filter_paraphrases(list(zip(subset, styles, outs)), nli=nli, nli_threshold=0.5)
Path('data/paraphrased').mkdir(parents=True, exist_ok=True)
write_jsonl('data/paraphrased/paraphrased_train.jsonl', kept)
stats['teacher'] = TEACHER; stats['nli_model'] = NLI_MODEL
Path('data/paraphrased/filter_stats.json').write_text(json.dumps(stats, indent=2))
print(json.dumps(stats, indent=2))
for e in kept[:2]:
    print('-' * 80); print(e['text'])

In [ ]:
from jobx.baselines.runs import push_results
push_results([Path('data/paraphrased/paraphrased_train.jsonl').resolve(),
              Path('data/paraphrased/filter_stats.json').resolve()])

## 3. Real postings: fetch + draft labels
The fetch picks the first candidate in `configs/real_data.yaml` whose declared licence is in the allowlist. **A human still has to confirm the licence** (see `docs/real_dataset.md`). Drafts go to a *private* dataset repo; verify them locally with `jobx verify`.

In [ ]:
import subprocess
if DO_REAL_DRAFTS:
    r = subprocess.run(['jobx', 'fetch-real'], capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])

In [ ]:
from jobx import prompts
from jobx.schema import parse_output, JobPosting
raw_path = Path('data/real/raw_postings.jsonl')
if DO_REAL_DRAFTS and raw_path.exists():
    raw = read_jsonl(raw_path)
    shots = prompts.select_few_shot(train, k=3)
    drafts = []
    for i in tqdm(range(0, len(raw), 4)):
        chunk = raw[i:i + 4]
        gens = generate(model, tok, [prompts.messages(r['text'], shots) for r in chunk], max_new_tokens=400)
        for r, g in zip(chunk, gens):
            pr = parse_output(g.text)
            label = pr.posting.to_json_dict() if pr.posting else JobPosting().to_json_dict()
            drafts.append({'id': r['id'], 'text': r['text'], 'label': label,
                           'meta': {'source': 'real', 'dataset': r['dataset'], 'row': r['row'],
                                    'licence': r['licence'], 'draft_model': TEACHER,
                                    'draft_schema_ok': pr.schema_ok}})
    write_jsonl('data/real/test_real_draft.jsonl', drafts)
    print(sum(d['meta']['draft_schema_ok'] for d in drafts), 'of', len(drafts), 'drafts schema-valid')
    push_results([Path('data/real/test_real_draft.jsonl').resolve(), Path('data/real/raw_meta.json').resolve()])
else:
    print('no real postings fetched; see the output above')

## Next
On your machine: `jobx pull` (downloads these files), `jobx verify` (≈30 min for a sample), then `jobx build-data` to merge the paraphrases and refresh `DATA_CARD.md`; commit and push. Then run `02_baselines` and `03_train_qlora`.